# Exercise 5: Optimise Under a Cap

Lab 27's first allocator reallocates a fixed weekly budget across the five Sedge Shoes channels with a greedy rule: it hands each small slice of budget to whichever channel's next slice adds the most expected revenue, with nothing to stop it pushing one channel far above its current spend or closing another altogether.
On the reference model's curves it closes YouTube and OOH, the kind of move Chapter 27 warns about: a plan that cuts a channel to nothing usually says more about an uncertain curve than about the channel.
This exercise asks you to write the bounded version Chapter 27 describes, in which every channel starts at a floor and no channel may be pushed past a cap.
You then check that the bounded plan spends the whole budget, respects every bound and still beats the current allocation.

In [1]:
import arviz as az
import pymc as pm

from mmm_book import RANDOM_SEED
from mmm_book.optimise import ResponseCurves, greedy_allocate
from mmm_book.plotting import mmm_style
from mmm_book.reference import national_aggregate, reference_model
from mmm_book.sedge import CHANNELS, generate_sedge

mmm_style()

## Fitting the reference model

This reuses the shared national-aggregate reference model from `mmm_book.reference`, the model Lab 15 builds with the priors Lab 17 chooses.
Sampling uses the settings every lab uses (four chains of 1,000 draws after 1,000 warm-up steps, target acceptance 0.95), and the cell checks the fit samples cleanly before anything is read from it.

In [2]:
df = generate_sedge()
nat = national_aggregate(df)

with reference_model(nat):
    idata = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        cores=1,
        target_accept=0.95,
        random_seed=RANDOM_SEED,
        progressbar=False,
    )

n_divergent = int(idata.sample_stats["diverging"].sum())
r_hat = float(az.summary(idata, kind="diagnostics")["r_hat"].max())
print(f"divergent transitions: {n_divergent}, largest r_hat: {r_hat:.3f}")

beta_mean = {ch: float(idata.posterior[f"beta_{ch}"].mean()) for ch in CHANNELS}

Initializing NUTS using jitter+adapt_diag...


Sequential sampling (4 chains in 1 job)


NUTS: [intercept, trend, season_a, season_b, omega_launch, delta_promo, gamma_log_price, gamma_distribution, gamma_stock, gamma_temperature, gamma_rainfall, gamma_category, beta_tv, beta_search, beta_meta, beta_youtube, beta_ooh, sigma]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 25 seconds.


divergent transitions: 0, largest r_hat: 1.004


## Response curves and the current allocation

`ResponseCurves` builds the curves Labs 26 and 27 use from the posterior-mean betas: a budget scales the channel's whole historical weekly plan, and Television's GRPs are valued at `TV_COST_PER_GRP` so every channel is allocated in pounds.
`curves.money_contribution(ch, money)` is the expected weekly contribution of channel `ch` at an average weekly spend of `money`.
The budget being reallocated is the current total, so the comparison is like for like.

In [3]:
curves = ResponseCurves(nat, beta_mean)
current = curves.current_money
total_budget = curves.total_budget

print(f"total weekly budget: {total_budget:,.0f} GBP")
for ch in CHANNELS:
    print(
        f"  {ch:<10}{current[ch]:>10,.0f} GBP  ->  "
        f"{curves.money_contribution(ch, current[ch]):>9,.0f} GBP a week"
    )

total weekly budget: 115,385 GBP
  tv            26,923 GBP  ->     64,423 GBP a week
  search        26,923 GBP  ->     83,593 GBP a week
  meta          28,846 GBP  ->     88,670 GBP a week
  youtube       15,385 GBP  ->     15,699 GBP a week
  ooh           17,308 GBP  ->     26,024 GBP a week


## Floors and caps

Suppose finance will not approve more than one and a half times a channel's current spend in a single reallocation, and the media team will not cut any channel below half of its current spend, so that contracts and presence survive while the evidence improves.
`floors` and `caps` encode those limits, in the same weekly pounds as `current`.

In [4]:
floors = {ch: current[ch] * 0.5 for ch in CHANNELS}
caps = {ch: current[ch] * 1.5 for ch in CHANNELS}
for ch in CHANNELS:
    print(f"  {ch:<10}floor {floors[ch]:>9,.0f}   cap {caps[ch]:>9,.0f}")

  tv        floor    13,462   cap    40,385
  search    floor    13,462   cap    40,385
  meta      floor    14,423   cap    43,269
  youtube   floor     7,692   cap    23,077
  ooh       floor     8,654   cap    25,962


## Task

Write `bounded_allocate`, the allocator Chapter 27 describes:

- start every channel at its floor
- divide the budget left over into `n_steps` equal slices
- give each slice to the channel whose next slice, `money_contribution(ch, alloc + step) - money_contribution(ch, alloc)`, adds the most, considering only channels for which `alloc[ch] + step <= caps[ch]`

Since the floors sum to less than the budget and the caps to more, every slice lands somewhere and the plan spends exactly the budget.
The stub below is Lab 27's first allocator, with no bounds at all.

In [5]:
n_steps = 4000


def bounded_allocate(curves, floors, caps, total, n_steps):
    # YOUR CODE HERE: start every channel at its floor and skip channels at their cap.
    # Stub: Lab 27's first allocator, which ignores floors and caps, so the checks below
    # will not pass yet.
    alloc = {ch: 0.0 for ch in CHANNELS}
    step = total / n_steps
    for _ in range(n_steps):
        gains = {
            ch: curves.money_contribution(ch, alloc[ch] + step)
            - curves.money_contribution(ch, alloc[ch])
            for ch in CHANNELS
        }
        alloc[max(gains, key=gains.get)] += step
    return alloc


plan = bounded_allocate(curves, floors, caps, total_budget, n_steps)

## Current against bounded plan

In [6]:
print(f"{'channel':<10}{'current':>10}{'floor':>10}{'plan':>10}{'cap':>10}")
for ch in CHANNELS:
    print(f"{ch:<10}{current[ch]:>10,.0f}{floors[ch]:>10,.0f}{plan[ch]:>10,.0f}{caps[ch]:>10,.0f}")

current_total = curves.total_contribution(current)
plan_total = curves.total_contribution(plan)
print(f"\ncurrent total contribution: {current_total:,.0f} GBP a week")
print(f"bounded plan contribution:  {plan_total:,.0f} GBP a week")
print(f"gain from reallocating within the bounds: {plan_total / current_total - 1:.1%}")

channel      current     floor      plan       cap
tv            26,923    13,462    60,375    40,385
search        26,923    13,462    26,942    40,385
meta          28,846    14,423    28,067    43,269
youtube       15,385     7,692         0    23,077
ooh           17,308     8,654         0    25,962

current total contribution: 278,409 GBP a week
bounded plan contribution:  297,195 GBP a week
gain from reallocating within the bounds: 6.7%


## Self-check

The last check compares your plan with `mmm_book.optimise.greedy_allocate`, the companion's own bounded allocator, given the same floors and caps.

In [7]:
reference_plan = greedy_allocate(curves, floors, caps, total=total_budget, n_steps=n_steps)

checks = {
    "every channel at or above its floor": all(plan[ch] >= floors[ch] - 1e-6 for ch in CHANNELS),
    "every channel at or below its cap": all(plan[ch] <= caps[ch] + 1e-6 for ch in CHANNELS),
    "plan spends the whole budget": abs(sum(plan.values()) - total_budget) < 1e-6,
    "plan beats the current allocation": plan_total > current_total,
    "plan matches greedy_allocate": all(
        abs(plan[ch] - reference_plan[ch]) < 1e-6 for ch in CHANNELS
    ),
}
for name, ok in checks.items():
    print(f"{name}: {'PASS' if ok else 'FAIL'}")
print("Self-check: PASS" if all(checks.values()) else "Self-check: FAIL")

every channel at or above its floor: FAIL
every channel at or below its cap: FAIL
plan spends the whole budget: PASS
plan beats the current allocation: PASS
plan matches greedy_allocate: FAIL
Self-check: FAIL
